# 01 — Extração histórica de propostas normativas

Extrai cinco anos de processos normativos, com campos mínimos para séries temporais e futura atribuição de temas.

Importa as bibliotecas utilizadas, define o endpoint GraphQL, o período da coleta, os tipos de proposição e a consulta que será enviada à API.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, time, requests, pandas as pd
ENDPOINT="https://legis.senado.leg.br/dadosabertos/graphql"
INICIO="2021-09-16"; FIM="2026-09-16"; LIMITE=10000
TIPOS=["PROJETO_LEI_ORDINARIA","PROJETO_LEI_COMPLEMENTAR","PROJETO_DECRETO_LEGISLATIVO","PROJETO_RESOLUCAO","MEDIDA_PROVISORIA","PROPOSTA_EMENDA_CONSTITUICAO","PROJETO_LEI_CONVERSAO","PROJETO_LEI"]
QUERY="""query($limit:Int!,$offset:Int!,$inicio:Date!,$fim:Date!,$tipos:[String!]) { processos(filter:{dataInicioApresentacao:$inicio,dataFimApresentacao:$fim,siglaTipoDocumento:$tipos},pageReq:{limit:$limit,offset:$offset,sortBy:[\"dataApresentacaoDocumento\"],order:[\"asc\"]}) { id identificacao ementa indexacao tipoDocumento dataApresentacaoDocumento urlDocumento } }"""

Percorre a API em páginas, valida os registros retornados, monta um DataFrame e salva os dados históricos e seus metadados em arquivos CSV e JSON.

In [ ]:
registros=[]
for offset in range(0,LIMITE,100):
    payload={"query":QUERY,"variables":{"limit":100,"offset":offset,"inicio":INICIO,"fim":FIM,"tipos":TIPOS}}
    corpo=requests.post(ENDPOINT,json=payload,timeout=30).json()
    if corpo.get("errors"): raise RuntimeError(corpo["errors"])
    pagina=corpo["data"]["processos"]; registros.extend(pagina)
    print(offset,len(pagina),len(registros))
    if len(pagina)<100: break
    time.sleep(.15)
df=pd.DataFrame(registros)
assert df.id.is_unique and df.ementa.fillna("").str.strip().ne("").all()
Path("../data/raw").mkdir(parents=True,exist_ok=True)
df.to_csv("../data/raw/propostas_normativas_20210916_a_20260916.csv",index=False,encoding="utf-8")
Path("../data/raw/propostas_normativas.metadata.json").write_text(json.dumps({"coletado_em_utc":datetime.now(timezone.utc).isoformat(),"registros":len(df),"inicio":INICIO,"fim":FIM,"tipos":TIPOS},ensure_ascii=False,indent=2),encoding="utf-8")
print(df.shape)